# Análise Exploratória dos dados

In [ ]:
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from google.colab import drive
import pandas as pd
import numpy as np

In [356]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [357]:
base_dir = '/content/drive/MyDrive/supervised-learning-studies/projeto'
file_path = os.path.join(base_dir, 'BD_filerCalcario_versao_1.xlsx')
file = pd.ExcelFile(file_path)
base_dir = '/content/drive/MyDrive/supervised-learning-studies/projeto/resistencia'
print(file.sheet_names)

['BD_ML_Resis', 'BD_ML_Slump_1', 'Dicionario_Variaveis']


In [358]:
df_resistencia = file.parse('BD_ML_Resis')
df_resistencia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599 entries, 0 to 598
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Autores/ano                  599 non-null    object 
 1   ID_Mistura                   599 non-null    object 
 2   Tipo_Cimento_Classe          599 non-null    object 
 3   Massa_Esp_Cimento_g_cm3      462 non-null    float64
 4   Finura_Cimento_Blaine_m2_kg  414 non-null    float64
 5   Massa_Esp_Filer_g_cm3        358 non-null    float64
 6   Massa_Esp_Areia_g_cm3        262 non-null    float64
 7   Massa_Esp_Brita_g_cm3        273 non-null    float64
 8   Cimento_kg_m3                599 non-null    float64
 9   Filer_Calcario_kg_m3         599 non-null    float64
 10  Agua_kg_m3                   599 non-null    float64
 11  Agregado_Miudo_Total_kg_m3   599 non-null    float64
 12  Agregado_Graudo_Total_kg_m3  599 non-null    float64
 13  Superplastificante_k

In [359]:
df_resistencia.Tipo_Cimento_Classe.sort_values(ascending=False).value_counts()

,count
Tipo_Cimento_Classe,
"CEM I 42,5 N",175
ASTM C 150 Type I,75
CP-V ARI,51
Type I OPC,50
"CEM I 42,5 R",41
OPC,40
Type I Portland,24
"P,I 42,5 (CEM I)",21
OPC (ASTM C150),21


In [360]:
autores = df_resistencia['Autores/ano'].unique()

for autor in autores:
    df_autor = df_resistencia[df_resistencia['Autores/ano'] == autor]
    print(f"{autor}, Número de amostras: {len(df_autor)} | {df_autor.shape[0]/len(df_resistencia)*100:.2f}%")

Dos Santos et al, 2024, Número de amostras: 15 | 2.50%
Guemmadi et al, 2009, Número de amostras: 26 | 4.34%
Meddah et al 2014, Número de amostras: 175 | 29.22%
A Morzouki 2016, Número de amostras: 35 | 5.84%
Bentz et al, 2015, Número de amostras: 11 | 1.84%
Feltrin 2018, Número de amostras: 36 | 6.01%
Gyu Don Moon 2017, Número de amostras: 24 | 4.01%
Hieu T Cam 2010, Número de amostras: 12 | 2.00%
Md Jahidul Islam 2025, Número de amostras: 15 | 2.50%
Diab et al, (2016), Número de amostras: 24 | 4.01%
Mohammed e Al-Numan (2024), Número de amostras: 15 | 2.50%
Leeuwen et al, (2016), Número de amostras: 50 | 8.35%
Ramezanianpour et al. (2009), Número de amostras: 75 | 12.52%
Bayan 2018, Número de amostras: 25 | 4.17%
Bonavetti et al. (2000)., Número de amostras: 9 | 1.50%
Tsivilis et al. (2003), Número de amostras: 10 | 1.67%
Sun e Chen 2018, Número de amostras: 21 | 3.51%
Abdul-Ghani et al, 2019, Número de amostras: 21 | 3.51%


In [361]:
dos_santos_mask = df_resistencia['Autores/ano'] == 'Dos Santos et al, 2024'
df_resistencia = df_resistencia[~dos_santos_mask]

In [362]:
df_resistencia.groupby("Autores/ano")['ID_Mistura'].nunique().sort_values(ascending=False)

,ID_Mistura
Autores/ano,
"Guemmadi et al, 2009",26
Meddah et al 2014,25
Ramezanianpour et al. (2009),15
Feltrin 2018,12
"Leeuwen et al, (2016)",10
Gyu Don Moon 2017,8
A Morzouki 2016,7
Sun e Chen 2018,7
"Abdul-Ghani et al, 2019",7


In [363]:
classe_encoder = LabelEncoder()
df_resistencia['cod_classe'] = classe_encoder.fit_transform(df_resistencia['Tipo_Cimento_Classe'])
df_resistencia['cod_classe'].value_counts()

,count
cod_classe,
6,175
0,75
17,50
7,41
13,40
10,36
18,24
15,21
14,21


In [367]:
# 1. Imputação inteligente: Preenche o Blaine vazio com a mediana da sua respectiva Classe
df_resistencia['Finura_Cimento_Blaine_m2_kg'] = df_resistencia.groupby('Classe_Cim_OpA')['Finura_Cimento_Blaine_m2_kg'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Rede de segurança: Se uma classe inteira não tiver nenhum dado de Blaine,
# usamos a mediana global do dataset para os retardatários.
mediana_global_blaine = df_resistencia['Finura_Cimento_Blaine_m2_kg'].median()
df_resistencia['Finura_Cimento_Blaine_m2_kg'] = df_resistencia['Finura_Cimento_Blaine_m2_kg'].fillna(mediana_global_blaine)

# Verificação final para garantir que zeramos os nulos
print(f"Valores nulos no Blaine após tratamento: {df_resistencia['Finura_Cimento_Blaine_m2_kg'].isna().sum()}")

Valores nulos no Blaine após tratamento: 0


In [372]:
df_resistencia.info()

<class 'pandas.core.frame.DataFrame'>
Index: 584 entries, 15 to 598
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Autores/ano                  584 non-null    object 
 1   ID_Mistura                   584 non-null    object 
 2   Tipo_Cimento_Classe          584 non-null    object 
 3   Massa_Esp_Cimento_g_cm3      447 non-null    float64
 4   Finura_Cimento_Blaine_m2_kg  584 non-null    float64
 5   Massa_Esp_Filer_g_cm3        358 non-null    float64
 6   Massa_Esp_Areia_g_cm3        247 non-null    float64
 7   Massa_Esp_Brita_g_cm3        258 non-null    float64
 8   Cimento_kg_m3                584 non-null    float64
 9   Filer_Calcario_kg_m3         584 non-null    float64
 10  Agua_kg_m3                   584 non-null    float64
 11  Agregado_Miudo_Total_kg_m3   584 non-null    float64
 12  Agregado_Graudo_Total_kg_m3  584 non-null    float64
 13  Superplastificante_kg_m3

In [327]:
df_resistencia["Usa_SP"] = df_resistencia["Usa_SP"].astype(bool)

In [328]:
df_resistencia.Usa_SP.dtype

dtype('bool')

In [333]:
exclude_columns = {'Autores/ano', 'ID_Mistura', 'Tipo_Cimento_Classe'}

In [341]:
df_resistencia.drop(columns=["ID_Mistura", "Tipo_Cimento_Classe"], inplace=True)

In [342]:
columns = df_resistencia.select_dtypes(include=['number']).columns

outliers_idx = {}
for column in columns:

    data = df_resistencia[column]
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_idx[column] = data[(data < lower_bound) | (data > upper_bound)].index.tolist()

for column, idx in sorted(outliers_idx.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{column}: {len(idx)} outliers")

D_Max: 192 outliers
Massa_Esp_Filer_kg_m3: 180 outliers
aglomerante_total: 131 outliers
s_fib: 131 outliers
hasselman_resiliencia_agua: 92 outliers
vol_areia: 90 outliers
ryshkewitch_inicial: 89 outliers
schiller_inicial: 89 outliers
Filer_D50_um: 88 outliers
vol_ar_aprisionado: 78 outliers
porosidade_volumetrica_inicial: 78 outliers
ryshkewitch_linearizado: 78 outliers
hasselman_fator_inicial: 78 outliers
aci_desvio_cimento: 76 outliers
Metodo_Emp_Cod: 71 outliers
vol_po_total: 61 outliers
Superplastificante_kg_m3: 55 outliers
vol_sp: 55 outliers
indice_vazios: 53 outliers
ryshkewitch_real: 53 outliers
hasselman_fator_real: 53 outliers
schiller_real: 53 outliers
margem_colapso_schiller: 53 outliers
taxa_sp_aglomerante: 52 outliers
volume_materiais: 50 outliers
taxa_sp_cimento: 49 outliers
hasselman_cinetico: 49 outliers
ryshkewitch_cinetico: 41 outliers
Agregado_Miudo_Total_kg_m3: 40 outliers
finos_total: 35 outliers
range_granulometrico: 30 outliers
Agregado_Graudo_Total_kg_m3: 25 ou

In [343]:
counter = Counter()

for column, idx_list in outliers_idx.items():
    counter.update(idx_list)

outliers_comuns = pd.DataFrame(
    counter.items(),
    columns=["idx", "n_outlier_cols"]
).sort_values(
    "n_outlier_cols",
    ascending=False
)

outliers_comuns.value_counts("n_outlier_cols").sort_index()

,count
n_outlier_cols,
1,46
2,88
3,157
4,59
5,9
6,43
7,15
8,32
9,1


In [344]:
limite = np.percentile(outliers_comuns["n_outlier_cols"], 90)

In [345]:
df_resistencia.drop(index=outliers_comuns[outliers_comuns["n_outlier_cols"] > limite]["idx"], inplace=True)
df_resistencia.shape

(533, 94)

In [ ]:
df_resistencia.columns, df_resistencia.shape

Index(['Finura_Cimento_Blaine_m2_kg', 'Cimento_kg_m3', 'Filer_Calcario_kg_m3',
       'Agua_kg_m3', 'Agregado_Miudo_Total_kg_m3',
       'Agregado_Graudo_Total_kg_m3', 'Superplastificante_kg_m3',
       'Relacao_Agua_Cimento', 'ln_Idade', 'Tipo_Molde_Cod', 'Metodo_Emp_Cod',
       'Usa_SP', 'Classe_Cim_OpA', 'Filer_D50_um', 'D_Max',
       'Resistencia_Compressao_MPa', 'cod_classe', 'familia_cod', 'f',
       'Massa_Esp_Cimento_kg_m3', 'Massa_Esp_Areia_kg_m3',
       'Massa_Esp_Brita_kg_m3', 'Massa_Esp_Filer_kg_m3', 'vol_agua',
       'vol_areia', 'vol_brita', 'volume_materiais', 'vol_ar_aprisionado',
       'porosidade_volumetrica_inicial', 'vol_solidos_totais', 'indice_vazios',
       'vol_agregados', 'vol_pasta', 'vol_po_total', 'pasta_agregado',
       'fracao_agregados', 'aglomerante_total', 'inv_a_c', 'fator_agua_po',
       'finos_total', 'taxa_filer_inerte', 'taxa_sp_cimento',
       'taxa_sp_aglomerante', 'range_granulometrico', 'idade_fator_agua',
       'parametro_feret', 'i

In [ ]:
df_resistencia.to_pickle(os.path.join(base_dir, f'df_resistencia_sem_feature_engineering.pkl'))

In [ ]:
# Limites de range das váriaveis numéricas
limites = {}
for column in df_resistencia.select_dtypes(include=['number']).columns:
    data = df_resistencia[column]
    lower_bound = data.min()
    upper_bound = data.max()
    limites[column] = (lower_bound, upper_bound)
print("Limites de range das variáveis numéricas:")
for column, (lower, upper) in limites.items():
    print(f"{column}: [{lower:.4f}, {upper:.4f}]")

Limites de range das variáveis numéricas:
Finura_Cimento_Blaine_m2_kg: [260.0000, 530.0000]
Cimento_kg_m3: [129.3000, 487.0000]
Filer_Calcario_kg_m3: [0.0000, 184.5000]
Agua_kg_m3: [134.0000, 205.0000]
Agregado_Miudo_Total_kg_m3: [595.0000, 1074.0000]
Agregado_Graudo_Total_kg_m3: [732.4000, 1249.0000]
Superplastificante_kg_m3: [0.0000, 4.0000]
Relacao_Agua_Cimento: [0.3600, 1.4300]
ln_Idade: [0.0000, 5.8999]
Tipo_Molde_Cod: [1.0000, 5.0000]
Metodo_Emp_Cod: [0.0000, 1.0000]
Classe_Cim_OpA: [32.0000, 63.0000]
Filer_D50_um: [0.0000, 72.0000]
D_Max: [12.5000, 30.0000]
Resistencia_Compressao_MPa: [1.0000, 71.0000]
cod_classe: [0.0000, 18.0000]
familia_cod: [0.0000, 7.0000]
Massa_Esp_Cimento_kg_m3: [3040.0000, 3270.0000]
Massa_Esp_Areia_kg_m3: [2570.0000, 2700.0000]
Massa_Esp_Brita_kg_m3: [2510.0000, 2650.0000]
Massa_Esp_Filer_kg_m3: [2550.0000, 2920.0000]
vol_agua: [0.1340, 0.2050]
vol_areia: [0.2215, 0.4086]
vol_brita: [0.2918, 0.4804]
volume_materiais: [0.9195, 1.0479]
vol_ar_aprisionado: